In [ ]:
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
kaggle_path = '/kaggle/input/datasets/tenus01/cifar-10'
if os.path.exists(os.path.join(kaggle_path, 'cifar-10-batches-py')):
    data_dir = kaggle_path
else:
    data_dir = os.path.dirname(kaggle_path)

print(f"Mounting Kaggle Dataset from: '{data_dir}'...")
# I uploaded the dataset to kaggle dir bcz it was taking a lot of time to fetch it agn for each run of expt. pls load the dataset by making download=True
# below or as you like it

#  Mathematical Transforms 
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # Free data augmentation!
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load to RAM
train_dataset = torchvision.datasets.CIFAR10(
    root=data_dir, 
    train=True, 
    download=False, 
    transform=transform
)

#  DataLoader
batch_size = 256 # Safe for T4/P100. Push to 512 if using L4.
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True,
    drop_last=True
)

print(f"Success! Loaded {len(train_dataset)} images into {len(train_loader)} batches.")

In [ ]:
import math
import torch.nn as nn

# ==========================================
# 1. THE COSINE NOISE SCHEDULE
# (Replaces the linear schedule for vastly better middle-step learning)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
timesteps = 1000

def get_cosine_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.02).to(device)

betas = get_cosine_schedule(timesteps)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0).to(device)

def add_noise(images, noise, t, alpha_cumprod):
    sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod[t])[:, None, None, None]
    sqrt_one_minus_alpha_cumprod = torch.sqrt(1 - alpha_cumprod[t])[:, None, None, None]
    return sqrt_alpha_cumprod * images + sqrt_one_minus_alpha_cumprod * noise

# ==========================================
# 2. UPGRADED UNET COMPONENTS
# ==========================================
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=time.device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class ResNetBlock(nn.Module):
    # UPGRADE: Injected Dropout (0.1)
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=8, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_channels))
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.gn1 = nn.GroupNorm(groups, out_channels)
        self.act1 = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.gn2 = nn.GroupNorm(groups, out_channels)
        self.act2 = nn.SiLU()
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.dropout(h) # Dropout applied directly to the math stream
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.shortcut(x)

class SelfAttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)
        self.ln = nn.LayerNorm([channels])

    def forward(self, x):
        b, c, h, w = x.shape
        x_flat = x.view(b, c, h * w).transpose(1, 2)
        x_norm = self.ln(x_flat)
        attn_out, _ = self.mha(x_norm, x_norm, x_norm)
        return (attn_out + x_flat).transpose(1, 2).view(b, c, h, w)

# ==========================================
# 3. THE UPGRADED ARCHITECTURE
# ==========================================
class DDPMUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, time_emb_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim), 
            nn.Linear(time_emb_dim, time_emb_dim), 
            nn.SiLU()
        )
        self.init_conv = nn.Conv2d(in_channels, 64, 3, padding=1)
        
        # DOWN PATH
        self.down1 = ResNetBlock(64, 64, time_emb_dim) 
        self.pool1 = nn.MaxPool2d(2) 
        
        self.down2 = ResNetBlock(64, 128, time_emb_dim) 
        self.attn_down = SelfAttentionBlock(128) # UPGRADE: Attention at 16x16
        self.pool2 = nn.MaxPool2d(2) 
        
        # BOTTLENECK (8x8)
        self.mid1 = ResNetBlock(128, 128, time_emb_dim)
        self.mid_attn = SelfAttentionBlock(128)
        self.mid2 = ResNetBlock(128, 128, time_emb_dim)
        
        # UP PATH
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest') 
        self.up1 = ResNetBlock(128 + 128, 128, time_emb_dim) 
        self.attn_up = SelfAttentionBlock(128) # UPGRADE: Attention at 16x16
        
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest') 
        self.up2 = ResNetBlock(128 + 64, 64, time_emb_dim) 
        
        self.final_conv = nn.Conv2d(64, out_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        
        # Down
        x1 = self.init_conv(x)
        d1 = self.down1(x1, t_emb)      # 32x32, 64ch
        
        d2 = self.down2(self.pool1(d1), t_emb) # 16x16, 128ch
        d2 = self.attn_down(d2)
        
        # Bottleneck
        m = self.mid1(self.pool2(d2), t_emb)   # 8x8, 128ch
        m = self.mid_attn(m)
        m = self.mid2(m, t_emb)
        
        # Up
        u1 = self.upsample1(m)
        u1 = torch.cat([u1, d2], dim=1) # Concat mid (128) and d2 (128)
        u1 = self.up1(u1, t_emb)        # 16x16, 128ch
        u1 = self.attn_up(u1)
        
        u2 = self.upsample2(u1)
        u2 = torch.cat([u2, d1], dim=1) # Concat u1 (128) and d1 (64)
        u2 = self.up2(u2, t_emb)        # 32x32, 64ch
        
        return self.final_conv(u2)

model = DDPMUNet().to(device)
print(f"Upgraded Model Built! Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#  EMA + AMP TRAINING LOOP (WITH VISUAL & WEIGHT CHECKPOINTS)
import os
import copy
import torch
from tqdm import tqdm
from torchvision.utils import save_image, make_grid

# ==========================================
# 1. DIRECTORY & EMA SETUP
# ==========================================
print("something started in this cell")
progress_dir = "./training_progress"
os.makedirs(progress_dir, exist_ok=True)

class EMA:
    def __init__(self, beta=0.995):
        self.beta = beta
    
    def step_ema(self, ema_model, current_model):
        for ema_param, param in zip(ema_model.parameters(), current_model.parameters()):
            if param.requires_grad:
                ema_param.data.mul_(self.beta).add_(param.data, alpha=1.0 - self.beta)

# Create the shadow model BEFORE compiling the main model
ema_model = copy.deepcopy(model).eval().to(device)
ema_model.requires_grad_(False)
ema = EMA(beta=0.995)

# ==========================================
# 2. PYTORCH COMPILE & AMP SETUP
# ==========================================
print("Compiling model for peak performance...")
try:
    compiled_model = torch.compile(model)
except Exception as e:
    print(f"torch.compile not fully supported, falling back. ({e})")
    compiled_model = model

optimizer = torch.optim.AdamW(compiled_model.parameters(), lr=2e-4)
loss_fn = torch.nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

epochs = 300
print(f"Initiating Training... Progress images and model checkpoints will be saved to '{progress_dir}' every 50 epochs.")

# ==========================================
# 3. TRAINING LOOP
# ==========================================

for epoch in range(epochs):
    compiled_model.train()
    epoch_loss = 0
    
    for step, (images, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")):
        images = images.to(device)
        current_batch = images.shape[0]
        
        t = torch.randint(0, timesteps, (current_batch,), device=device).long()
        true_noise = torch.randn_like(images, device=device)
        noisy_images = add_noise(images, true_noise, t, alpha_cumprod)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_noise = compiled_model(noisy_images, t)
            loss = loss_fn(pred_noise, true_noise)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Update the shadow EMA model every single step
        if hasattr(compiled_model, '_orig_mod'):
            ema.step_ema(ema_model, compiled_model._orig_mod)
        else:
            ema.step_ema(ema_model, compiled_model)
            
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.6f}")

    # ==========================================
    # 4. CHECKPOINTING (Every 50 Epochs)
    # ==========================================
    if (epoch + 1) % 50 == 0:
        print(f"--> Generating progress grid and saving weights for Epoch {epoch+1}...")
        
        # --- A. Save the Model Weights ---
        checkpoint_path = os.path.join(progress_dir, f"ema_model_epoch_{epoch+1:03d}.pth")
        torch.save(ema_model.state_dict(), checkpoint_path)
        
        # --- B. Generate the Visual Grid ---
        ema_model.eval() 
        with torch.no_grad():
            x_gen = torch.randn((10, 3, 32, 32)).to(device) 
            
            for i in tqdm(reversed(range(timesteps)), total=timesteps, desc="Sampling", leave=False):
                t_gen = torch.full((10,), i, device=device, dtype=torch.long)
                pred_noise_gen = ema_model(x_gen, t_gen)
                
                a = alphas[t_gen][:, None, None, None]
                a_hat = alpha_cumprod[t_gen][:, None, None, None]
                b = betas[t_gen][:, None, None, None]
                
                z = torch.randn_like(x_gen) if i > 0 else torch.zeros_like(x_gen)
                x_gen = 1 / torch.sqrt(a) * (x_gen - ((1 - a) / (torch.sqrt(1 - a_hat))) * pred_noise_gen) + torch.sqrt(b) * z
            
            x_gen = (x_gen.clamp(-1, 1) + 1) / 2
            
            grid = make_grid(x_gen, nrow=5, padding=2, normalize=True)
            img_save_path = os.path.join(progress_dir, f"grid_epoch_{epoch+1:03d}.png")
            save_image(grid, img_save_path)

# Save final weights to the main directory
torch.save(ema_model.state_dict(), "ddpm_cifar10_EMA_weights.pth")

if hasattr(compiled_model, '_orig_mod'):
    torch.save(compiled_model._orig_mod.state_dict(), "ddpm_cifar10_RAW_weights.pth")
else:
    torch.save(compiled_model.state_dict(), "ddpm_cifar10_RAW_weights.pth")
    
print("\nTraining Complete! Final weights secured.")

In [ ]:
import os
from torchvision.utils import save_image

# Settings
total_images_needed = 1000
gen_batch_size = 200 

@torch.no_grad()
def sample_images(diffusion_model, n_samples):
    diffusion_model.eval()
    x = torch.randn((n_samples, 3, 32, 32)).to(device)
    
    for i in tqdm(reversed(range(timesteps)), total=timesteps, desc=f"Sampling Batch", leave=False):
        t = torch.full((n_samples,), i, device=device, dtype=torch.long)
        predicted_noise = diffusion_model(x, t)
        
        alpha = alphas[t][:, None, None, None]
        alpha_hat = alpha_cumprod[t][:, None, None, None]
        beta = betas[t][:, None, None, None]
        
        noise = torch.randn_like(x) if i > 0 else torch.zeros_like(x)
        x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        
    return (x.clamp(-1, 1) + 1) / 2

gen_dir = "./evaluation_data/generated"
os.makedirs(gen_dir, exist_ok=True)

batches = total_images_needed // gen_batch_size
remainder = total_images_needed % gen_batch_size

print(f"\nGenerating {total_images_needed} images using EMA weights...")
img_counter = 0

# Use the ema_model for generation!
for b in range(batches):
    gen_imgs = sample_images(ema_model, n_samples=gen_batch_size) 
    for i in range(gen_batch_size):
        save_image(gen_imgs[i], os.path.join(gen_dir, f"img_{img_counter:05d}.png"))
        img_counter += 1

if remainder > 0:
    gen_imgs = sample_images(ema_model, n_samples=remainder)
    for i in range(remainder):
        save_image(gen_imgs[i], os.path.join(gen_dir, f"img_{img_counter:05d}.png"))
        img_counter += 1

print(f"Generation Complete! Dataset saved to '{gen_dir}'")

In [ ]:
# EVALUATION
import os
import torch
import torchvision
import torchvision.transforms as transforms
import torch_fidelity
from torch.utils.data import Dataset

kaggle_path = '/kaggle/input/datasets/tenus01/cifar-10'
data_dir = kaggle_path if os.path.exists(os.path.join(kaggle_path, 'cifar-10-batches-py')) else os.path.dirname(kaggle_path)
#pls adjust the path acc to your sys
base_dataset = torchvision.datasets.CIFAR10(
    root=data_dir, 
    train=True, 
    download=False, 
    transform=transforms.PILToTensor() 
)

class ImageOnlyDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        return self.dataset[idx][0]

evaluation_dataset = ImageOnlyDataset(base_dataset)

def evaluate_model(generated_images_dir: str, real_data):
    print("Executing memory-safe torch_fidelity diagnostic report...")
    
    metrics = torch_fidelity.calculate_metrics(
        input1=generated_images_dir,
        input2=real_data, 
        cuda=True,
        isc=True,
        fid=True,
        prc=False,          # using only fid and IS right now we'll put prec and recall once we get good values for atleast fid and IS
        save_cpu_ram=True,  
        verbose=False 
    )
    
    fid_score = metrics.get('frechet_inception_distance', 'N/A')
    is_mean = metrics.get('inception_score_mean', 'N/A')
    is_std = metrics.get('inception_score_std', 'N/A')

    print("=============================================")
    print("             DIAGNOSTIC REPORT")
    print("=============================================")
    if isinstance(fid_score, float): print(f"FID:       {fid_score:.4f}  (Lower is better)")
    if isinstance(is_mean, float): print(f"IS:        {is_mean:.4f} ± {is_std:.4f}  (Higher is better)")
    print("=============================================")

gen_dir = "./evaluation_data/generated"

evaluate_model(gen_dir, evaluation_dataset)
#I got an fid 0f 56 and IS of 6.3 with this model